In [1]:
from google.colab import files
import pandas as pd
import zipfile

subidos_senamhi = files.upload()
nombre_zip_senamhi = next(iter(subidos_senamhi))

with zipfile.ZipFile(nombre_zip_senamhi) as paquete:
    archivos_estaciones = sorted(
        nombre for nombre in paquete.namelist()
        if nombre.startswith("consolidados/")
        and nombre.endswith(".csv")
    )

    if len(archivos_estaciones) != 4:
        raise ValueError(
            "Se esperaban cuatro CSV en la carpeta consolidados."
        )

    tablas = []

    for nombre in archivos_estaciones:
        tabla = pd.read_csv(
            paquete.open(nombre),
            encoding="utf-8-sig",
            dtype=str,
            keep_default_na=False
        )
        tabla["ARCHIVO_ORIGEN"] = nombre
        tablas.append(tabla)

meteo_original = pd.concat(tablas, ignore_index=True)

resumen_meteo = (
    meteo_original
    .groupby(["CODIGO_ESTACION", "ESTACION"], as_index=False)
    .agg(
        REGISTROS=("FECHA", "size"),
        PRIMERA_FECHA=("FECHA", "min"),
        ULTIMA_FECHA=("FECHA", "max")
    )
)

print("Estaciones cargadas:", len(resumen_meteo))
print("Registros horarios:", len(meteo_original))
display(resumen_meteo)

Saving SENAMHI_4_estaciones_2022_2024_preliminar.zip to SENAMHI_4_estaciones_2022_2024_preliminar (1).zip
Estaciones cargadas: 4
Registros horarios: 102217


,CODIGO_ESTACION,ESTACION,REGISTROS,PRIMERA_FECHA,ULTIMA_FECHA
0,111286,CARABAYLLO,25335,2022-01-01,2024-12-31
1,112181,CAMPO DE MARTE,26110,2022-01-01,2024-12-31
2,472A0766,VILLA MARIA DEL TRIUNFO,24520,2022-01-01,2024-12-18
3,472AC278,VON HUMBOLDT,26252,2022-01-01,2024-12-31


In [2]:
# Trabajar sobre una copia y conservar los datos originales.
meteo_revision = meteo_original.copy()

meteo_revision["TEMP_NUM"] = pd.to_numeric(
    meteo_revision["TEMPERATURA_C"], errors="coerce"
)
meteo_revision["HR_NUM"] = pd.to_numeric(
    meteo_revision["HUMEDAD_RELATIVA_PCT"], errors="coerce"
)

# Indicadores iniciales de revisión, no un control completo de calidad.
meteo_revision["TEMP_MAYOR_100"] = (
    meteo_revision["TEMP_NUM"] > 100
)
meteo_revision["HR_CERO"] = (
    meteo_revision["HR_NUM"] == 0
)

resumen_calidad = (
    meteo_revision.groupby("ESTACION", as_index=False)
    .agg(
        REGISTROS=("FECHA", "size"),
        TEMP_MIN=("TEMP_NUM", "min"),
        TEMP_MAX=("TEMP_NUM", "max"),
        TEMP_FALTANTE=("TEMP_NUM", lambda s: s.isna().sum()),
        HR_FALTANTE=("HR_NUM", lambda s: s.isna().sum()),
        TEMP_MAYOR_100=("TEMP_MAYOR_100", "sum"),
        HR_CERO=("HR_CERO", "sum")
    )
)

print("REVISIÓN INICIAL POR ESTACIÓN")
display(resumen_calidad)

print("REGISTROS CON TEMPERATURA SUPERIOR A 100 °C")
display(
    meteo_revision.loc[
        meteo_revision["TEMP_MAYOR_100"],
        ["ESTACION", "FECHA", "HORA",
         "TEMPERATURA_C", "HUMEDAD_RELATIVA_PCT"]
    ]
)

REVISIÓN INICIAL POR ESTACIÓN


,ESTACION,REGISTROS,TEMP_MIN,TEMP_MAX,TEMP_FALTANTE,HR_FALTANTE,TEMP_MAYOR_100,HR_CERO
0,CAMPO DE MARTE,26110,12.1,331.8,162,131,6,30
1,CARABAYLLO,25335,11.6,33.9,13,13,0,0
2,VILLA MARIA DEL TRIUNFO,24520,11.1,31.6,13,14,0,0
3,VON HUMBOLDT,26252,8.7,330.0,5,56,1,0


REGISTROS CON TEMPERATURA SUPERIOR A 100 °C


,ESTACION,FECHA,HORA,TEMPERATURA_C,HUMEDAD_RELATIVA_PCT
30039,CAMPO DE MARTE,2022-07-17,14:00,233.9,
30789,CAMPO DE MARTE,2022-08-18,12:00,234,81
34997,CAMPO DE MARTE,2023-02-13,21:00,331.8,
36515,CAMPO DE MARTE,2023-04-18,08:00,151.7,100
40958,CAMPO DE MARTE,2023-10-20,20:00,305.5,
49050,CAMPO DE MARTE,2024-09-22,06:00,234,
77157,VON HUMBOLDT,2022-02-19,23:00,330,


In [8]:
meteo_revision["FECHA_HORA"] = pd.to_datetime(
    meteo_revision["FECHA"] + " " + meteo_revision["HORA"],
    errors="coerce"
)

casos = meteo_revision.loc[
    meteo_revision["TEMP_MAYOR_100"],
    ["CODIGO_ESTACION", "FECHA_HORA"]
]

for _, caso in casos.iterrows():
    misma_estacion = (
        meteo_revision["CODIGO_ESTACION"]
        == caso["CODIGO_ESTACION"]
    )

    horas_cercanas = meteo_revision["FECHA_HORA"].between(
        caso["FECHA_HORA"] - pd.Timedelta(hours=1),
        caso["FECHA_HORA"] + pd.Timedelta(hours=1)
    )

    display(
        meteo_revision.loc[
            misma_estacion & horas_cercanas,
            [
                "ESTACION", "FECHA_HORA",
                "TEMPERATURA_C", "HUMEDAD_RELATIVA_PCT",
                "TEMP_MAYOR_100"
            ]
        ].sort_values("FECHA_HORA")
    )

,ESTACION,FECHA_HORA,TEMPERATURA_C,HUMEDAD_RELATIVA_PCT,TEMP_MAYOR_100
30039,CAMPO DE MARTE,2022-07-17 14:00:00,233.9,,True
30040,CAMPO DE MARTE,2022-07-17 15:00:00,16.1,,False


,ESTACION,FECHA_HORA,TEMPERATURA_C,HUMEDAD_RELATIVA_PCT,TEMP_MAYOR_100
30788,CAMPO DE MARTE,2022-08-18 11:00:00,15.4,82,False
30789,CAMPO DE MARTE,2022-08-18 12:00:00,234,81,True
30790,CAMPO DE MARTE,2022-08-18 13:00:00,15.8,81,False


,ESTACION,FECHA_HORA,TEMPERATURA_C,HUMEDAD_RELATIVA_PCT,TEMP_MAYOR_100
34996,CAMPO DE MARTE,2023-02-13 20:00:00,23.5,84,False
34997,CAMPO DE MARTE,2023-02-13 21:00:00,331.8,,True
34998,CAMPO DE MARTE,2023-02-13 22:00:00,23,88,False


,ESTACION,FECHA_HORA,TEMPERATURA_C,HUMEDAD_RELATIVA_PCT,TEMP_MAYOR_100
36514,CAMPO DE MARTE,2023-04-18 07:00:00,,1,False
36515,CAMPO DE MARTE,2023-04-18 08:00:00,151.7,100,True
36516,CAMPO DE MARTE,2023-04-18 09:00:00,24.9,100,False


,ESTACION,FECHA_HORA,TEMPERATURA_C,HUMEDAD_RELATIVA_PCT,TEMP_MAYOR_100
40957,CAMPO DE MARTE,2023-10-20 19:00:00,18.6,93,False
40958,CAMPO DE MARTE,2023-10-20 20:00:00,305.5,,True
40959,CAMPO DE MARTE,2023-10-20 21:00:00,18.1,95,False


,ESTACION,FECHA_HORA,TEMPERATURA_C,HUMEDAD_RELATIVA_PCT,TEMP_MAYOR_100
49049,CAMPO DE MARTE,2024-09-22 05:00:00,14.8,95,False
49050,CAMPO DE MARTE,2024-09-22 06:00:00,234,,True
49051,CAMPO DE MARTE,2024-09-22 07:00:00,14.6,97,False


,ESTACION,FECHA_HORA,TEMPERATURA_C,HUMEDAD_RELATIVA_PCT,TEMP_MAYOR_100
77156,VON HUMBOLDT,2022-02-19 22:00:00,19.5,80,False
77157,VON HUMBOLDT,2022-02-19 23:00:00,330,,True
77158,VON HUMBOLDT,2022-02-20 00:00:00,19.4,83,False


In [10]:
# Copia de trabajo: conserva todos los registros y valores originales
meteo_qc = meteo_revision.copy()

# Temperatura para cálculos preliminares
meteo_qc["TEMP_USO_PROVISIONAL"] = meteo_qc["TEMP_NUM"].mask(
    meteo_qc["TEMP_MAYOR_100"]
)

# Documentar por qué se apartó cada valor
meteo_qc["MOTIVO_EXCLUSION_TEMP"] = ""

meteo_qc.loc[
    meteo_qc["TEMP_MAYOR_100"],
    "MOTIVO_EXCLUSION_TEMP"
] = "Temperatura >100 °C; no utilizable; pendiente contraste con fuente"

print("Filas conservadas:", len(meteo_qc))
print(
    "Temperaturas apartadas:",
    int(meteo_qc["TEMP_MAYOR_100"].sum())
)

display(
    meteo_qc.loc[
        meteo_qc["TEMP_MAYOR_100"],
        [
            "ESTACION", "FECHA_HORA", "TEMPERATURA_C",
            "TEMP_USO_PROVISIONAL", "MOTIVO_EXCLUSION_TEMP"
        ]
    ]
)

Filas conservadas: 102217
Temperaturas apartadas: 7


,ESTACION,FECHA_HORA,TEMPERATURA_C,TEMP_USO_PROVISIONAL,MOTIVO_EXCLUSION_TEMP
30039,CAMPO DE MARTE,2022-07-17 14:00:00,233.9,NaN,Temperatura >100 °C; no utilizable; pendiente ...
30789,CAMPO DE MARTE,2022-08-18 12:00:00,234,NaN,Temperatura >100 °C; no utilizable; pendiente ...
34997,CAMPO DE MARTE,2023-02-13 21:00:00,331.8,NaN,Temperatura >100 °C; no utilizable; pendiente ...
36515,CAMPO DE MARTE,2023-04-18 08:00:00,151.7,NaN,Temperatura >100 °C; no utilizable; pendiente ...
40958,CAMPO DE MARTE,2023-10-20 20:00:00,305.5,NaN,Temperatura >100 °C; no utilizable; pendiente ...
49050,CAMPO DE MARTE,2024-09-22 06:00:00,234,NaN,Temperatura >100 °C; no utilizable; pendiente ...
77157,VON HUMBOLDT,2022-02-19 23:00:00,330,NaN,Temperatura >100 °C; no utilizable; pendiente ...


In [11]:
# Identificar valores fuera del intervalo porcentual
meteo_qc["HR_FUERA_RANGO"] = (
    meteo_qc["HR_NUM"].notna()
    & ~meteo_qc["HR_NUM"].between(0, 100)
)

# Bandera exploratoria: no modifica ni excluye valores
meteo_qc["HR_BAJA_REVISION"] = meteo_qc["HR_NUM"].between(0, 5)

resumen_hr = (
    meteo_qc.groupby("ESTACION", as_index=False)
    .agg(
        HR_FALTANTE=("HR_NUM", lambda s: s.isna().sum()),
        HR_CERO=("HR_NUM", lambda s: s.eq(0).sum()),
        HR_ENTRE_0_Y_5=("HR_BAJA_REVISION", "sum"),
        HR_FUERA_RANGO=("HR_FUERA_RANGO", "sum")
    )
)

display(resumen_hr)

casos_hr = meteo_qc.loc[
    meteo_qc["HR_FUERA_RANGO"] | meteo_qc["HR_BAJA_REVISION"],
    [
        "ESTACION", "FECHA_HORA", "HUMEDAD_RELATIVA_PCT",
        "TEMP_USO_PROVISIONAL", "HR_FUERA_RANGO"
    ]
].sort_values(["ESTACION", "FECHA_HORA"])

with pd.option_context("display.max_rows", None):
    display(casos_hr)

,ESTACION,HR_FALTANTE,HR_CERO,HR_ENTRE_0_Y_5,HR_FUERA_RANGO
0,CAMPO DE MARTE,131,30,106,0
1,CARABAYLLO,13,0,0,0
2,VILLA MARIA DEL TRIUNFO,14,0,1381,0
3,VON HUMBOLDT,56,0,0,0


,ESTACION,FECHA_HORA,HUMEDAD_RELATIVA_PCT,TEMP_USO_PROVISIONAL,HR_FUERA_RANGO
36294,CAMPO DE MARTE,2023-04-09 03:00:00,1,NaN,False
36295,CAMPO DE MARTE,2023-04-09 04:00:00,1,NaN,False
36296,CAMPO DE MARTE,2023-04-09 05:00:00,1,NaN,False
36392,CAMPO DE MARTE,2023-04-13 05:00:00,1,NaN,False
36393,CAMPO DE MARTE,2023-04-13 06:00:00,1,NaN,False
36394,CAMPO DE MARTE,2023-04-13 07:00:00,1,NaN,False
36395,CAMPO DE MARTE,2023-04-13 08:00:00,1,NaN,False
36405,CAMPO DE MARTE,2023-04-13 18:00:00,1,NaN,False
36406,CAMPO DE MARTE,2023-04-13 19:00:00,1,NaN,False
36409,CAMPO DE MARTE,2023-04-13 22:00:00,1,NaN,False


In [12]:
resumen_hr_baja_anual = (
    meteo_qc.loc[meteo_qc["HR_BAJA_REVISION"]]
    .assign(ANIO=lambda tabla: tabla["FECHA_HORA"].dt.year)
    .groupby(["ESTACION", "ANIO"], as_index=False)
    .agg(
        REGISTROS_HR_BAJA=("HR_NUM", "size"),
        HR_MIN=("HR_NUM", "min"),
        HR_MAX=("HR_NUM", "max"),
        PRIMER_CASO=("FECHA_HORA", "min"),
        ULTIMO_CASO=("FECHA_HORA", "max")
    )
)

display(resumen_hr_baja_anual)

,ESTACION,ANIO,REGISTROS_HR_BAJA,HR_MIN,HR_MAX,PRIMER_CASO,ULTIMO_CASO
0,CAMPO DE MARTE,2023,106,0.0,1.0,2023-04-09 03:00:00,2023-05-03 09:00:00
1,VILLA MARIA DEL TRIUNFO,2023,172,0.8,1.2,2023-10-04 01:00:00,2023-12-20 07:00:00
2,VILLA MARIA DEL TRIUNFO,2024,1209,0.8,1.3,2024-01-02 03:00:00,2024-12-17 07:00:00


In [13]:
import io

archivo_contraste = files.upload()
contenido = next(iter(archivo_contraste.values()))

portal_octubre = pd.read_csv(
    io.BytesIO(contenido),
    skiprows=1,
    skipfooter=2,
    engine="python",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)

portal_octubre["FECHA_HORA"] = pd.to_datetime(
    portal_octubre["AÑO / MES / DÍA"] + " " + portal_octubre["HORA"],
    format="%d/%m/%Y %H:%M"
)

assert len(portal_octubre) == 744
assert portal_octubre["FECHA_HORA"].is_unique

es_octubre_vmt = (
    meteo_qc["CODIGO_ESTACION"].eq("472A0766")
    & meteo_qc["FECHA_HORA"].dt.strftime("%Y-%m").eq("2023-10")
)

hr_portal = portal_octubre.set_index("FECHA_HORA")["HUMEDAD (%)"]

meteo_qc.loc[es_octubre_vmt, "HR_PORTAL_OCT2023"] = (
    meteo_qc.loc[es_octubre_vmt, "FECHA_HORA"].map(hr_portal)
)

meteo_qc["HR_SD_EN_PORTAL_OCT2023"] = (
    es_octubre_vmt
    & meteo_qc["HR_PORTAL_OCT2023"].eq("S/D")
)

print("Humedades marcadas S/D en el portal:",
      int(meteo_qc["HR_SD_EN_PORTAL_OCT2023"].sum()))

Saving 472A0766_202310.csv to 472A0766_202310.csv
Humedades marcadas S/D en el portal: 126


In [14]:
# Distinguir los motivos de apartamiento
meteo_qc["MOTIVO_APARTADO_HR"] = ""

meteo_qc.loc[
    meteo_qc["HR_BAJA_REVISION"],
    "MOTIVO_APARTADO_HR"
] = "Valor bajo pendiente de validacion"

meteo_qc.loc[
    meteo_qc["HR_FUERA_RANGO"],
    "MOTIVO_APARTADO_HR"
] = "Fuera de 0-100 %"

meteo_qc.loc[
    meteo_qc["HR_SD_EN_PORTAL_OCT2023"],
    "MOTIVO_APARTADO_HR"
] = "S/D confirmado en CSV del portal"

# Columna provisional para cálculos; conserva HR_NUM y el texto original
meteo_qc["HR_APARTADA"] = meteo_qc["MOTIVO_APARTADO_HR"].ne("")

meteo_qc["HR_USO_PROVISIONAL"] = meteo_qc["HR_NUM"].mask(
    meteo_qc["HR_APARTADA"]
)

resumen_hr_preliminar = (
    meteo_qc.groupby("ESTACION", as_index=False)
    .agg(
        FALTANTES_ORIGINALES=("HR_NUM", lambda s: s.isna().sum()),
        APARTADAS_EN_REVISION=("HR_APARTADA", "sum"),
        DISPONIBLES_PROVISIONALES=("HR_USO_PROVISIONAL", "count")
    )
)

display(resumen_hr_preliminar)

,ESTACION,FALTANTES_ORIGINALES,APARTADAS_EN_REVISION,DISPONIBLES_PROVISIONALES
0,CAMPO DE MARTE,131,106,25873
1,CARABAYLLO,13,0,25322
2,VILLA MARIA DEL TRIUNFO,14,1381,23125
3,VON HUMBOLDT,56,0,26196


In [15]:
# Calendario esperado, conservando las horas tal como vienen de la fuente
calendario_horario = pd.date_range(
    start="2022-01-01",
    end="2025-01-01",
    freq="h",
    inclusive="left"
)

# Comprobar las claves antes de contar
assert meteo_qc["FECHA_HORA"].notna().all(), "Hay fechas inválidas"
assert not meteo_qc.duplicated(
    ["CODIGO_ESTACION", "FECHA_HORA"]
).any(), "Hay horas duplicadas por estación"
assert meteo_qc["FECHA_HORA"].isin(
    calendario_horario
).all(), "Hay fechas fuera del periodo o de la cuadrícula horaria"

resumen_horas = []
tablas_horas_ausentes = []

for (codigo, estacion), tabla in meteo_qc.groupby(
    ["CODIGO_ESTACION", "ESTACION"]
):
    presentes = pd.DatetimeIndex(tabla["FECHA_HORA"])
    ausentes = calendario_horario.difference(presentes)

    resumen_horas.append({
        "ESTACION": estacion,
        "HORAS_ESPERADAS": len(calendario_horario),
        "HORAS_CON_FILA": len(presentes),
        "HORAS_SIN_FILA": len(ausentes)
    })

    tablas_horas_ausentes.append(pd.DataFrame({
        "CODIGO_ESTACION": codigo,
        "ESTACION": estacion,
        "FECHA_HORA": ausentes
    }))

resumen_cobertura_horaria = pd.DataFrame(resumen_horas)
horas_ausentes = pd.concat(tablas_horas_ausentes, ignore_index=True)

display(resumen_cobertura_horaria)

,ESTACION,HORAS_ESPERADAS,HORAS_CON_FILA,HORAS_SIN_FILA
0,CARABAYLLO,26304,25335,969
1,CAMPO DE MARTE,26304,26110,194
2,VILLA MARIA DEL TRIUNFO,26304,24520,1784
3,VON HUMBOLDT,26304,26252,52


In [16]:
meteo_qc["FECHA_DIA"] = meteo_qc["FECHA_HORA"].dt.normalize()

# Ambas variables disponibles en la misma hora
meteo_qc["TEMP_Y_HR_DISPONIBLES"] = (
    meteo_qc["TEMP_USO_PROVISIONAL"].notna()
    & meteo_qc["HR_USO_PROVISIONAL"].notna()
)

conteos_diarios = (
    meteo_qc.groupby(["CODIGO_ESTACION", "FECHA_DIA"])
    .agg(
        HORAS_CON_FILA=("FECHA_HORA", "size"),
        HORAS_TEMP=("TEMP_USO_PROVISIONAL", "count"),
        HORAS_HR=("HR_USO_PROVISIONAL", "count"),
        HORAS_AMBAS=("TEMP_Y_HR_DISPONIBLES", "sum")
    )
)

dias = pd.date_range("2022-01-01", "2024-12-31", freq="D")

indice_completo = pd.MultiIndex.from_product(
    [meteo_qc["CODIGO_ESTACION"].unique(), dias],
    names=["CODIGO_ESTACION", "FECHA_DIA"]
)

# Los ceros indican cantidad de horas disponibles, no temperatura ni humedad
cobertura_diaria = (
    conteos_diarios.reindex(indice_completo, fill_value=0)
    .reset_index()
    .merge(
        meteo_qc[["CODIGO_ESTACION", "ESTACION"]].drop_duplicates(),
        on="CODIGO_ESTACION",
        validate="many_to_one"
    )
)

resumen_dias = (
    cobertura_diaria.groupby("ESTACION", as_index=False)
    .agg(
        DIAS_ESPERADOS=("FECHA_DIA", "size"),
        TEMP_24H=("HORAS_TEMP", lambda s: s.eq(24).sum()),
        HR_24H=("HORAS_HR", lambda s: s.eq(24).sum()),
        AMBAS_24H=("HORAS_AMBAS", lambda s: s.eq(24).sum()),
        AMBAS_1_A_23H=("HORAS_AMBAS", lambda s: s.between(1, 23).sum()),
        AMBAS_0H=("HORAS_AMBAS", lambda s: s.eq(0).sum())
    )
)

display(resumen_dias)

,ESTACION,DIAS_ESPERADOS,TEMP_24H,HR_24H,AMBAS_24H,AMBAS_1_A_23H,AMBAS_0H
0,CAMPO DE MARTE,1096,994,992,991,95,10
1,CARABAYLLO,1096,1013,1013,1013,59,24
2,VILLA MARIA DEL TRIUNFO,1096,952,794,794,250,52
3,VON HUMBOLDT,1096,1053,1050,1050,44,2


In [17]:
# Medias de las horas disponibles, conservadas para revisión
medias_observadas = (
    meteo_qc.groupby(["CODIGO_ESTACION", "FECHA_DIA"], as_index=False)
    .agg(
        TEMP_MEDIA_HORAS_DISPONIBLES=("TEMP_USO_PROVISIONAL", "mean"),
        HR_MEDIA_HORAS_DISPONIBLES=("HR_USO_PROVISIONAL", "mean")
    )
)

senamhi_diario_preliminar = cobertura_diaria.merge(
    medias_observadas,
    on=["CODIGO_ESTACION", "FECHA_DIA"],
    how="left",
    validate="one_to_one"
)

# Medias para días con las 24 observaciones de cada variable
senamhi_diario_preliminar["TEMP_MEDIA_24H"] = (
    senamhi_diario_preliminar["TEMP_MEDIA_HORAS_DISPONIBLES"]
    .where(senamhi_diario_preliminar["HORAS_TEMP"].eq(24))
)

senamhi_diario_preliminar["HR_MEDIA_24H"] = (
    senamhi_diario_preliminar["HR_MEDIA_HORAS_DISPONIBLES"]
    .where(senamhi_diario_preliminar["HORAS_HR"].eq(24))
)

resumen_base_diaria = (
    senamhi_diario_preliminar.groupby("ESTACION", as_index=False)
    .agg(
        DIAS_CONSERVADOS=("FECHA_DIA", "size"),
        DIAS_CON_MEDIA_TEMP_24H=("TEMP_MEDIA_24H", "count"),
        DIAS_CON_MEDIA_HR_24H=("HR_MEDIA_24H", "count")
    )
)

print("Filas estación-día:", len(senamhi_diario_preliminar))
display(resumen_base_diaria)

Filas estación-día: 4384


,ESTACION,DIAS_CONSERVADOS,DIAS_CON_MEDIA_TEMP_24H,DIAS_CON_MEDIA_HR_24H
0,CAMPO DE MARTE,1096,994,992
1,CARABAYLLO,1096,1013,1013
2,VILLA MARIA DEL TRIUNFO,1096,952,794
3,VON HUMBOLDT,1096,1053,1050


In [19]:
from pathlib import Path
import zipfile

tablas_respaldo = {
    "senamhi_horario_original.csv": meteo_original,
    "senamhi_horario_revision_preliminar.csv": meteo_qc,
    "senamhi_diario_preliminar.csv": senamhi_diario_preliminar,
    "senamhi_horas_ausentes.csv": horas_ausentes,
    "resumen_cobertura_horaria.csv": resumen_cobertura_horaria,
    "resumen_cobertura_diaria.csv": resumen_dias,
    "resumen_revision_humedad.csv": resumen_hr_preliminar,
    "contraste_portal_vmt_octubre_2023.csv": portal_octubre
}

notas = """
RESPALDO SENAMHI — PREPARACION PRELIMINAR

Periodo solicitado: 2022-2024.
Estaciones: Carabayllo, Campo de Marte, Villa Maria del Triunfo
y Von Humboldt. Su representatividad distrital sigue pendiente.

Se conservaron las 102217 filas recibidas y los valores originales.
Se apartaron 7 temperaturas superiores a 100 C, sin reconstruirlas.
Este filtro no constituye una validacion completa de temperatura.

Se apartaron provisionalmente 1487 humedades entre 0 y 5 %:
- 126 aparecen como S/D en el CSV del portal para VMT, octubre 2023.
- 1361 siguen pendientes de validacion.
El limite de 5 % es un filtro exploratorio, no una norma oficial.

Se identificaron 2999 horas sin fila respecto al calendario esperado.
Los valores faltantes o apartados no se sustituyeron por ceros.
No se realizo imputacion ni interpolacion.

La base diaria conserva 4384 filas: 1096 dias por estacion.
TEMP_MEDIA_24H y HR_MEDIA_24H requieren 24 horas disponibles
de su respectiva variable. Es un criterio inicial revisable.
Las medias de horas disponibles se conservan por separado.
Los ceros en columnas de conteo indican cantidades de observaciones.

Fechas y horas conservadas tal como fueron recibidas.
Zona horaria pendiente de confirmacion antes del cruce sanitario.
No se ha realizado asignacion de estaciones a distritos.

El SINADEF preparado previamente termina el 05/05/2024.
El cruce posterior debe respetar la disponibilidad temporal comun.

Estos archivos son preliminares y no equivalen a datos
completamente validados ni a una base definitiva para DLNM.
Los codigos de estacion deben leerse como texto.
Los campos vacios de mediciones representan datos no disponibles.
"""

ruta_zip = Path("SENAMHI_preparacion_preliminar_respaldo_v1.zip")
fuente_zip = Path(nombre_zip_senamhi)

assert fuente_zip.exists(), "No se encuentra el ZIP original cargado"

with zipfile.ZipFile(
    ruta_zip, "w", compression=zipfile.ZIP_DEFLATED
) as respaldo:

    for nombre, tabla in tablas_respaldo.items():
        respaldo.writestr(
            "tablas/" + nombre,
            tabla.to_csv(index=False, na_rep="").encode("utf-8-sig")
        )

    respaldo.writestr(
        "LEEME_SENAMHI_preliminar.txt",
        notas.encode("utf-8")
    )

    respaldo.write(
        fuente_zip,
        arcname="fuentes/" + fuente_zip.name
    )

    respaldo.writestr(
        "fuentes/472A0766_202310.csv",
        next(iter(archivo_contraste.values()))
    )

with zipfile.ZipFile(ruta_zip) as comprobacion:
    assert comprobacion.testzip() is None, "Error de integridad del ZIP"

print("Respaldo generado con", len(tablas_respaldo), "CSV y notas.")
files.download(str(ruta_zip))

Respaldo generado con 8 CSV y notas.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>